In [84]:
rho = 0.8
n = 9
topologyfile = f"../../../evaltopologyfiles/dring_80_64.edgelist"
nswitches = 80
netpathfile = f"npfiles/netpath_dring_80_64_fatpaths.np"
# topologyfile = "../test_dragonfly/tpfiles/df_p2_a3_h1.edgelist"
# nswitches = 12
# netpathfile = f"npfiles/df_p2_a3_h1_fatpaths.np"

In [72]:
# sanity check
import random
import networkx as nx
# random.seed(2)

V = range(nswitches)
E = list()
with open(topologyfile, 'r') as f:
    lines = f.readlines()
    for line in lines:
        tokens = line.split("->")
        E.append((int(tokens[0]), int(tokens[1])))

perm = random.sample(V, len(V))
perm_map = dict(zip(V, perm))

E_layer = list()
for (u,v) in E:
    if perm_map[u]<perm_map[v] and random.random()<0.8: # <1
        E_layer.append((u, v))
# print(E_layer)
print(len(E_layer))
G = nx.Graph()
G.add_nodes_from(V)
G.add_edges_from(E_layer)
if not nx.is_connected(G):
    print("Sanity check failed: generated layer is not connected")
else:   
    print("Sanity check passed: generated layer is connected")

443
Sanity check passed: generated layer is connected


In [81]:
# Generate layers, by the FatPaths paper / Listing 1
import random
import networkx as nx

V = range(nswitches)
# Generate E from topology file
E = list()
with open(topologyfile, 'r') as f:
    lines = f.readlines()
    for line in lines:
        tokens = line.split("->")
        E.append((int(tokens[0]), int(tokens[1])))
# print(E)

P = list()
L = list()
for i in range(n-1):
    seed = 0
    while True:
        random.seed(seed)

        perm = random.sample(V, len(V))
        perm_map = dict(zip(V, perm))
        P.append(perm_map)
        
        E_layer = list()
        for (u,v) in E:
            if perm_map[u]<perm_map[v] and random.random()<0.8:
                E_layer.append((u, v))

        # Check whether E_layer is connected; if not, regenerate
        # print(len(E_layer))
        G = nx.Graph()
        G.add_nodes_from(V)
        G.add_edges_from(E_layer)
        if nx.is_connected(G):
            print(f"layer {i} done")
            L.append(E_layer)
            seed += 1
            break
        seed += 1
# print(L)

layer 0 done
layer 1 done
layer 2 done
layer 3 done
layer 4 done
layer 5 done
layer 6 done
layer 7 done


In [ ]:
# Pick paths

def find_path_min_overlap(G, source, target, used_edges, max_extra_hops=1):
    try:
        shortest_len = nx.shortest_path_length(G, source, target)
    except nx.NetworkXNoPath:
        return None
    
    best_path = None
    best_overlap = float('inf')
    
    for cutoff in range(shortest_len, shortest_len + max_extra_hops + 1):
        for path in nx.all_simple_paths(G, source, target, cutoff=cutoff):
            path_edges = set(zip(path[:-1], path[1:]))
            overlap = len(path_edges & used_edges)
            if overlap < best_overlap:
                best_overlap = overlap
                best_path = path
                if best_overlap == 0:
                    return best_path  # perfect path found
    return best_path


net_paths_rack_based = list()
for i in range(nswitches):
    net_paths_rack_based.append(list())
    for j in range(nswitches):
        net_paths_rack_based[i].append(list())


paths_per_layer = []
used_edges_per_pair = {(i,j): set() for i in V for j in V if i != j}

for layer_idx, E_layer in enumerate(L):
    G = nx.Graph()
    G.add_nodes_from(V)
    G.add_edges_from(E_layer)
    
    layer_paths = {}  # {(i,j): path}
    
    for i in V:
        for j in V:
            print(f"Layer {layer_idx}, finding path from {i} to {j}")
            if i != j:
                used_edges = used_edges_per_pair[(i,j)]
                path = find_path_min_overlap(G, i, j, used_edges, max_extra_hops=2)
                if path is not None:
                    layer_paths[(i,j)] = path
                    used_edges_per_pair[(i,j)].update(zip(path[:-1], path[1:]))
    
    paths_per_layer.append(layer_paths)
    for (i,j), path in layer_paths.items():
        if path not in net_paths_rack_based[i][j]:
            net_paths_rack_based[i][j].append(path)


In [ ]:
# # Pick paths
# net_paths_rack_based = list()
# for i in range(nswitches):
#     net_paths_rack_based.append(list())
#     for j in range(nswitches):
#         net_paths_rack_based[i].append(list())


# def best_path_no_overlap(G, i, j, mypaths, max_extra_hops=3):
#     try:
#         shortest_len = nx.shortest_path_length(G, i, j)
#     except nx.NetworkXNoPath:
#         return None
    
#     # We’ll allow paths up to (shortest_len + max_extra_hops)
#     for cutoff in range(shortest_len, shortest_len + max_extra_hops + 1):
#         candidates = list(nx.all_simple_paths(G, i, j, cutoff=cutoff))
#         if not candidates:
#             continue

#         # Rank candidates by overlap
#         best = None
#         best_score = float('inf')
#         for path in candidates:
#             score = min_overlap(path, mypaths)
#             if score < best_score:
#                 best_score = score
#                 best = path

#         # If perfect non-overlap found, return immediately
#         if best_score == 0:
#             return best
        
#         # Otherwise, keep this best-in-class as fallback
#         fallback = best
    
#     # If we finish the loop without finding zero-overlap,
#     # return the best we saw (with minimal overlap)
#     return fallback

# def min_overlap(path, mypaths):
#     """Compute how many nodes overlap with already chosen paths."""
#     path_set = set(path)
#     min_score = float('inf')
#     if not mypaths:
#         return 0
#     for p in mypaths:
#         overlap = len(path_set.intersection(p))
#         min_score = min(min_score, overlap)
#     return min_score


# for i in range(nswitches):
#     for j in range(nswitches):
#         print(f"Finding paths from {i} to {j}")
#         mypaths = []
#         if i != j:
#             for E_layer in L:
#                 G = nx.Graph()
#                 G.add_edges_from(E_layer)

#                 best = best_path_no_overlap(G, i, j, mypaths)
#                 if best:
#                     mypaths.append(best)
#                 else:
#                     print(f"ERROR: No path found from {i} to {j}")

#         net_paths_rack_based[i][j].append(mypaths)

In [91]:
# Write netpathfile
with open(netpathfile,'w') as f:
    for i in range(nswitches):
        for j in range(nswitches):
            if i==j:
                f.write(f"{i} {j} 0\n")
            else:
                paths_rack_based = net_paths_rack_based[i][j]
                f.write(f"{i} {j} {len(paths_rack_based)}\n")
                for path in paths_rack_based:
                    for ihop in range(1,len(path)):
                        f.write(f" {path[ihop-1]}->{path[ihop]}")
                    f.write("\n")